# Book Rating Distributions

A book averaging 3.0 stars could mean two completely different things:

- Everyone thought it was mediocre
- Half the readers loved it (5★), half hated it (1★)

Same average. Completely different story.

The **shape** of the ratings tells you what the number alone can't. That shape is called a **distribution**, and reading it is one of the most immediately useful things you can learn in statistics.

This notebook covers the four most common shapes in book rating data, how to generate them in code, and what they reveal.

## Prerequisites

- Python, NumPy, matplotlib — you've used them before
- You've seen a histogram
- No prior stats knowledge needed

## What is a distribution?

A distribution describes the **shape** of a dataset. It answers one question:

> *If I pick one value at random from this dataset, what am I likely to get?*

As a programmer, you've already built one. This is a distribution:

```python
from collections import Counter
counts = Counter(ratings)
total = sum(counts.values())
prob  = {k: v / total for k, v in counts.items()}
```

That `prob` dict is a discrete probability distribution. For every possible rating value, it tells you how likely you are to see it.

The formal version — a **probability density function (PDF)** — is the same idea made smooth and continuous. Think of it as your `Counter` histogram with infinite data and infinitely small bins.

### The two numbers that define a simple distribution

- **Mean (μ)** — where the centre of the data is. The "if I had to guess one value" answer.
- **Standard deviation (σ)** — how spread out the data is. Roughly: the average distance of each point from the mean. Tight σ = everyone agrees. Wide σ = mixed opinions.

That's all you need to read most distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8')

# Simulate 5000 ratings for a typical well-liked book (mean 3.8, std 0.8)
np.random.seed(42)
simulated = np.random.normal(loc=3.8, scale=0.8, size=5000)
simulated = np.clip(simulated, 1, 5)  # ratings are bounded [1, 5]

fig, ax = plt.subplots(figsize=(9, 4))

# The histogram IS the distribution — each bar is a Counter bucket
ax.hist(simulated, bins=40, density=True,
        color='steelblue', alpha=0.55, edgecolor='white',
        label='Histogram of 5,000 simulated ratings')

# The PDF curve is a smooth mathematical description of the same shape
x = np.linspace(1, 5, 500)
ax.plot(x, norm.pdf(x, 3.8, 0.8), 'steelblue', linewidth=2.5,
        label='PDF — same shape, smoothed')

ax.axvline(simulated.mean(), color='navy', linestyle='--', alpha=0.7,
           label=f'Mean = {simulated.mean():.2f}')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Density')
ax.set_title('Histogram vs PDF — the curve is just the histogram made smooth')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean: {simulated.mean():.3f}  (target 3.8)")
print(f"Std:  {simulated.std():.3f}  (target 0.8)")

## The Four Shapes

| Shape | What it looks like | What it means for a book |
|---|---|---|
| **Normal** | Single peak, symmetric | Most readers broadly agree |
| **Bimodal** | Two peaks at opposite extremes | Two camps — love it or hate it |
| **Uniform** | Flat — no peak anywhere | Completely polarised across all ratings |
| **Right-skewed** | Peak near 4–5★, tail dragging left | Mostly liked, minority disliked it |

Each one tells you something about the *audience*, not just the book.

### Shape 1: Normal (Bell Curve)

The classic shape. Most reviewers land in the middle, with fewer at the extremes. When you see this, the population broadly agrees — the rating is a reliable signal.

`norm.pdf(x, loc, scale)` takes two parameters:
- `loc` = mean (where the peak is)
- `scale` = standard deviation (how wide the bell is)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.linspace(1, 5, 500)

y = norm.pdf(x, loc=3.8, scale=0.8)  # peak at 3.8, moderate spread
ax.plot(x, y, color='steelblue', linewidth=3)
ax.fill_between(x, y, alpha=0.12, color='steelblue')
ax.axvline(3.8, color='navy', linestyle='--', alpha=0.6, label='Mean (μ = 3.8)')

# Show ±1 std: where ~68% of ratings fall
ax.axvspan(3.8 - 0.8, 3.8 + 0.8, alpha=0.08, color='blue',
           label='±1σ — where ~68% of ratings fall')

ax.set_title('Normal Distribution — typical non-controversial book', fontsize=13, fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Density')
ax.set_xlim(1, 5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Shape 2: Bimodal (Two Peaks)

Two humps, one at each extreme. This is what you see with politically or ideologically charged books — readers self-select into two camps before they even open the book.

Technically: a bimodal distribution is the **sum of two normal distributions** with different means. Think of it as two distinct sub-populations whose ratings you're mixing into one dataset.

```python
# Two groups:
#   Group A (52%): mean 1.3, std 0.75  — the haters
#   Group B (48%): mean 4.7, std 0.65  — the fans
combined = 0.52 * norm.pdf(x, 1.3, 0.75) + 0.48 * norm.pdf(x, 4.7, 0.65)
```

This is a real pattern — you can look up the rating histogram for any culture-war book and see exactly this shape.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.linspace(1, 5, 500)

# Two overlapping normals — one per camp of readers
group_hate = norm.pdf(x, loc=1.3, scale=0.75)   # 52% of readers
group_love = norm.pdf(x, loc=4.7, scale=0.65)   # 48% of readers
y_bimodal  = 0.52 * group_hate + 0.48 * group_love

ax.plot(x, y_bimodal, color='crimson', linewidth=3)
ax.fill_between(x, y_bimodal, alpha=0.12, color='crimson')

# Show each component
ax.plot(x, 0.52 * group_hate, 'r--', alpha=0.5, linewidth=1.5, label='Haters (52%)')
ax.plot(x, 0.48 * group_love, 'b--', alpha=0.5, linewidth=1.5, label='Fans (48%)')

ax.set_title('Bimodal Distribution — polarised book (e.g. ideological/political)', fontsize=13, fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Density')
ax.set_xlim(1, 5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Shape 3: Uniform

Every rating is equally likely. This is rare in practice — it means the book provokes no shared reaction at all. When you see it, it's often a sign of a niche or highly subjective work, or a book with a very mixed audience that has no coherent view.

It's also a useful baseline: if you see deviation from flat, you know there's *signal* in the ratings.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.linspace(1, 5, 500)

y_uniform = np.ones_like(x) * 0.25  # every value equally likely

ax.plot(x, y_uniform, color='seagreen', linewidth=3)
ax.fill_between(x, y_uniform, alpha=0.12, color='seagreen')
ax.set_title('Uniform Distribution — no consensus whatsoever', fontsize=13, fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Density')
ax.set_xlim(1, 5)
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Shape 4: Right-Skewed (Positive Skew)

The peak is shifted toward the high end, with a longer tail pulling left. This is the most common shape for popular books — most people like it, but a minority doesn't.

Note the naming convention, which is counterintuitive: **right-skewed** (or positively skewed) means the *tail* points right — but for ratings, the tail pointing left means the peak is on the high end. For book ratings, a peak at 4–5★ with some drag toward 1★ is called **left-skewed** or negatively skewed in strict stats terms. In common usage it's often called "right-skewed" colloquially. Don't worry about the terminology for now — just read the shape.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.linspace(1, 5, 500)

# Shift the mean toward the high end, let the std create the leftward tail
y_skewed = norm.pdf(x, loc=4.1, scale=0.75)

ax.plot(x, y_skewed, color='darkorchid', linewidth=3)
ax.fill_between(x, y_skewed, alpha=0.12, color='darkorchid')
ax.axvline(4.1, color='purple', linestyle='--', alpha=0.6, label='Mean (4.1★)')
ax.set_title('Skewed Distribution — broadly well-liked book', fontsize=13, fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Density')
ax.set_xlim(1, 5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Side-by-side: all four shapes

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 9))
x = np.linspace(1, 5, 500)

configs = [
    (axs[0,0], norm.pdf(x, 3.8, 0.8),
     'Normal — broad consensus',             'steelblue'),
    (axs[0,1], 0.52 * norm.pdf(x, 1.3, 0.75) + 0.48 * norm.pdf(x, 4.7, 0.65),
     'Bimodal — two camps',                  'crimson'),
    (axs[1,0], np.ones_like(x) * 0.25,
     'Uniform — no consensus',               'seagreen'),
    (axs[1,1], norm.pdf(x, 4.1, 0.75),
     'Skewed — broadly liked, some dislike', 'darkorchid'),
]

for ax, y, title, color in configs:
    ax.plot(x, y, color=color, linewidth=3)
    ax.fill_between(x, y, alpha=0.12, color=color)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Star Rating')
    ax.set_ylabel('Density')
    ax.set_xlim(1, 5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Formal Notation

Now that you've seen what the shapes look like, here's the math that defines them.

The **normal distribution** PDF is:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{1}{2}\left(\frac{x - \mu}{\sigma}\right)^2}$$

Where:
- μ (mu) = mean — the centre of the bell
- σ (sigma) = standard deviation — the width of the bell
- The fraction out front is just a normalising constant so the area under the curve equals 1

The exponent `-(x - μ)²` is key: it's zero at the mean (peak), and grows as x moves away — so the function decays away from centre. The σ in the denominator controls how fast it decays.

In code: `norm.pdf(x, loc=μ, scale=σ)`

A **bimodal** distribution has no single formula — it's a weighted sum of two normals:

$$f(x) = w_1 \cdot \mathcal{N}(x; \mu_1, \sigma_1) + w_2 \cdot \mathcal{N}(x; \mu_2, \sigma_2) \quad \text{where } w_1 + w_2 = 1$$

The weights (0.52, 0.48 in our example) are the relative sizes of the two sub-populations.

## Exercises

Try these to build intuition. Modify the code cells above.

**1. Tighten and widen the normal**  
Change `scale=0.8` to `scale=0.3` and then `scale=1.5`. Watch the bell change width. What does a very narrow bell say about readers?

**2. Move the bimodal peaks closer together**  
Change `loc=1.3` to `loc=2.0` and `loc=4.7` to `loc=4.0`. At what point do the two humps merge into one?

**3. Make a trimodal distribution**  
Add a third normal component with its own weight. What kind of book might produce three rating clusters?

**4. Simulate a real dataset**  
Pick a book on Amazon or Goodreads and read the star breakdown (usually shown as a bar chart). Simulate it with `np.random.choice` and weights, then plot the histogram. Which of the four shapes does it most resemble?

## Real-World Connection

**In recommendation systems:**  
Average rating hides distribution shape. A system that only stores the mean loses the bimodal signal entirely — it can't tell you "half the people hated this book". Real recommender systems often store the full rating histogram, not just the mean.

**In A/B testing:**  
When you measure user behaviour (session length, clicks, revenue), the *shape* of the distribution matters as much as the average. A normal distribution lets you use t-tests. A bimodal one means you have two user segments and should be analysing them separately.

**In anomaly detection:**  
A product that suddenly shifts from a right-skewed to a bimodal distribution is a signal — something changed. Distribution shape over time is a useful monitoring feature.

**In ML feature engineering:**  
Distributions tell you how to preprocess. A normal distribution can go straight into a linear model. A heavily skewed one benefits from a log transform. Knowing the shape is step one.

## Summary

- A **distribution** describes the shape of a dataset — how frequently each value appears
- It's the smooth, continuous version of a histogram (`Counter` with infinite data)
- Two numbers define the simplest distributions: **mean** (centre) and **standard deviation** (spread)
- **Normal**: one peak, symmetric — the audience broadly agrees
- **Bimodal**: two peaks — two distinct camps in the audience; the same average can hide this completely
- **Uniform**: flat — no consensus, every rating equally likely
- **Skewed**: peak at one end with a tail — mostly one opinion with a minority dissenting
- The `scipy.stats.norm.pdf(x, loc=μ, scale=σ)` call generates a normal PDF for any x
- A bimodal distribution is just two normals added together with weights summing to 1